<a href="https://colab.research.google.com/github/JevinMakwana/edge-ai-baby-cry-detection/blob/Task-1/1_dataset_generation/audio_samples/generate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!apt-get -y install ffmpeg
%cd /content
!git clone -b Task-1 https://github.com/JevinMakwana/edge-ai-baby-cry-detection.git
%cd /content/edge-ai-baby-cry-detection
# !pip install -q -r requirements.txt
# %cd /content/edge-ai-baby-cry-detection/1_dataset_generation/audio_samples

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
/content
fatal: destination path 'edge-ai-baby-cry-detection' already exists and is not an empty directory.
/content/edge-ai-baby-cry-detection


In [27]:
!apt-get -y install ffmpeg build-essential python3-dev
%cd /content/edge-ai-baby-cry-detection

!python -m pip install -q --upgrade pip setuptools wheel
!python -m pip uninstall -y numpy
!python -m pip install -q "numpy>=1.26.0,<2.1"

!python -m pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!python -m pip install -q transformers>=4.30.0 soundfile soxr
!python -m pip install -q audioldm==0.1.1
!python -m pip install -q jedi==0.19.1

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
The following additional packages will be installed:
  javascript-common libjs-sphinxdoc libjs-underscore python3.10-dev
Suggested packages:
  apache2 | lighttpd | httpd
The following NEW packages will be installed:
  javascript-common libjs-sphinxdoc libjs-underscore python3-dev
  python3.10-dev
0 upgraded, 5 newly installed, 0 to remove and 37 not upgraded.
Need to get 508 kB/797 kB of archives.
After this operation, 1,257 kB of additional disk space will be used.
Ign:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3.10-dev amd64 3.10.12-1~22.04.14
Err:1 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 python3.10-dev amd64 3.10.12-1~22.04.14
  404  Not Found [IP: 91.189.91.81 80]
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/

In [28]:
!python -c "import numpy, torch, audioldm, soundfile; print('OK', numpy.__version__)"

OK 2.0.2


In [29]:
%cd /content/edge-ai-baby-cry-detection/1_dataset_generation/audio_samples

/content/edge-ai-baby-cry-detection/1_dataset_generation/audio_samples


In [30]:
from audioldm import text_to_audio, build_model
import os, soundfile as sf
import ast
from pathlib import Path
import torch
import numpy as np
import os

In [31]:
def patch_audioldm_for_cpu():
    if torch.cuda.is_available():
        return

    from audioldm.latent_diffusion.ddim import DDIMSampler

    def register_buffer_cpu_safe(self, name, attr):
        if isinstance(attr, torch.Tensor):
            target_device = getattr(self.model, "device", torch.device("cpu"))
            if attr.device != target_device:
                attr = attr.to(target_device)
        setattr(self, name, attr)

    DDIMSampler.register_buffer = register_buffer_cpu_safe


patch_audioldm_for_cpu()

# Load model once (expensive)
ldm = build_model(model_name="audioldm-s-full")

def load_prompts_from_txt(file_path, variable_name="prompts"):
    file_path = Path(file_path)
    content = None
    for encoding in ("utf-8", "utf-8-sig", "utf-16"):
        try:
            content = file_path.read_text(encoding=encoding)
            break
        except UnicodeDecodeError:
            continue

    if content is None:
        raise UnicodeDecodeError(
            "prompt_loader",
            b"",
            0,
            1,
            f"Unsupported encoding in {file_path}",
        )

    module = ast.parse(content, filename=str(file_path))

    for node in module.body:
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id == variable_name:
                    return ast.literal_eval(node.value)

    raise ValueError(f"Variable '{variable_name}' not found in {file_path}")


def resolve_prompts_dir():
    search_roots = []
    if "__file__" in globals():
        search_roots.append(Path(__file__).resolve().parent)
    search_roots.append(Path.cwd().resolve())

    for root in search_roots:
        for base in [root, *root.parents]:
            prompts_path = base / "prompts"
            if prompts_path.is_dir():
                return prompts_path

    raise FileNotFoundError(
        "Could not locate the 'prompts' directory. Run the notebook from the project root "
        "or place it within the project tree."
    )


prompts_dir = resolve_prompts_dir()

baby_cry_prompts = load_prompts_from_txt(prompts_dir / "baby_cry_prompts.txt")
background_prompts = load_prompts_from_txt(prompts_dir / "background_noise_prompts.txt")

Load AudioLDM: %s audioldm-s-full
DiffusionWrapper has 185.04 M params.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def generate_batch(prompts, output_dir, samples_per_prompt=3):
    os.makedirs(output_dir, exist_ok=True)
    for i, prompt in enumerate(prompts):
        print(f"Generating: {prompt}")
        waveforms = text_to_audio(
            ldm,
            text=prompt,
            seed=42,
            duration=5.0,
            guidance_scale=2.5,
            n_candidate_gen_per_text=samples_per_prompt
        )
        for j, wav in enumerate(waveforms):
            # Convert to numpy if needed
            if hasattr(wav, 'numpy'):
                wav = wav.numpy()
            else:
                wav = np.array(wav, dtype=np.float32)

            # Ensure mono (1D)
            if wav.ndim > 1:
                wav = wav.squeeze()

            # Ensure float32
            if wav.dtype != np.float32:
                wav = wav.astype(np.float32)

            filename = os.path.join(output_dir, f"{i:03d}_{j}.wav")

            print(f"  Writing: shape={wav.shape}, dtype={wav.dtype}, min={wav.min():.3f}, max={wav.max():.3f}")
            sf.write(filename, wav, samplerate=16000)
            print(f"  Saved: {filename} [{j+1}/{samples_per_prompt}]")

# Fixed paths (no duplicate audio_samples)
generate_batch(baby_cry_prompts, "/content/edge-ai-baby-cry-detection/1_dataset_generation/audio_samples/baby_cry")
generate_batch(background_prompts, "/content/edge-ai-baby-cry-detection/1_dataset_generation/audio_samples/background_noise")

Generating: Baby Crying
Generate audio using text Baby Crying


DDIM Sampler: 100%|██████████| 200/200 [00:24<00:00,  8.27it/s]


  Writing: shape=(81952,), dtype=float32, min=-0.300, max=0.301
  Saved: /content/edge-ai-baby-cry-detection/1_dataset_generation/audio_samples/baby_cry/000_0.wav [1/3]
Generating: Baby crying in bedroom
Generate audio using text Baby crying in bedroom


DDIM Sampler: 100%|██████████| 200/200 [00:24<00:00,  8.00it/s]


  Writing: shape=(81952,), dtype=float32, min=-0.405, max=0.374
  Saved: /content/edge-ai-baby-cry-detection/1_dataset_generation/audio_samples/baby_cry/001_0.wav [1/3]
Generating: Baby crying loudly
Generate audio using text Baby crying loudly


...

DDIM Sampler: 100%|██████████| 200/200 [00:24<00:00,  8.25it/s]


  Writing: shape=(81952,), dtype=float32, min=-0.943, max=0.970
  Saved: /content/edge-ai-baby-cry-detection/1_dataset_generation/audio_samples/baby_cry/018_0.wav [1/3]
Generating: Screaming baby
Generate audio using text Screaming baby


DDIM Sampler: 100%|██████████| 200/200 [00:24<00:00,  8.25it/s]


  Writing: shape=(81952,), dtype=float32, min=-0.850, max=0.889
  Saved: /content/edge-ai-baby-cry-detection/1_dataset_generation/audio_samples/baby_cry/019_0.wav [1/3]
Generating: Howling baby
Generate audio using text Howling baby


DDIM Sampler:  80%|████████  | 160/200 [00:19<00:04,  8.27it/s]